# 3a. BE vs SC Grid-Search Cross-Validation

**Purpose**: Classify each animal as BE or SC using the manuscript's
grid-search CV procedure on the update matrix.

**Protocol** (matching manuscript):
1. Expert sessions: last 50% of Uniform sessions with accuracy ≥ 70%
2. Pool trials, split into 2 folds by session blocks
3. Grid search: σ_percep × A_repulsion × model params → ~8,000 combos
4. 64 seeds → 64 test errors per model per animal
5. ANOVA: compare BE vs SC error distributions

**Data flow**: Cluster pickles → load → ANOVA → visualisation → parameter report

**Sections**:
1. Setup
2. Load cluster results
3. ANOVA model comparison
4. Per-animal comparison plots
5. Winner summary
6. Best-fit parameters
7. Empirical vs model update matrices
8. Parameter distributions across seeds
9. Save

## 1. Setup

In [1]:
import sys, os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from behav_utils.data.loading import load_experiment
from behav_utils.data.selection import select_sessions
from behav_utils.analysis.update_matrix import (
    compute_update_matrix_from_sessions, matrix_error,
)
from behav_utils.plotting.update_matrix import plot_phase_update_matrices
from behav_utils.plotting.styles import apply_style

from analysis.cv_utils import (
    load_cv_pickles, summarise_loaded_results,
    build_long_df, run_anova, build_summary_table,
    get_best_seed_params, format_params, extract_param_df,
)
from analysis.grid_search import _simulate_um
from plotting.cv import (
    plot_cv_comparison, plot_winner_summary,
    plot_param_distributions,
)

apply_style()
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

Imports OK


In [2]:
CONFIG_PATH = Path('../config.yaml')
CV_RESULTS_DIR = Path('../results/cv')
OUTPUT_DIR = Path('../results/cv/figures')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Expert session criteria (must match cluster job)
EXPERT_MIN_ACCURACY = 0.70
EXPERT_LAST_FRACTION = 0.50
N_BINS = 8

## 2. Load Cluster Results

Cluster job (`submit_cv.sh` → `run_cv_single.py`) produces one pickle
per animal per seed: `cv_{ANIMAL}_seed{NNN}.pkl`.
`gather_cv_results.py` merges these. `load_cv_pickles` loads them.

In [3]:
all_results = load_cv_pickles(str(CV_RESULTS_DIR))
summarise_loaded_results(all_results)

FileNotFoundError: No pickle files found matching ../results/cv/cv_*_seed*.pkl

In [ ]:
long_df = build_long_df(all_results)
print(f'Long DataFrame: {len(long_df)} rows')
print(f'Animals: {long_df["animal_id"].nunique()}')
print(f'Seeds per animal per model: '
      f'{long_df.groupby(["animal_id", "model"]).size().unique()}')
print()
long_df.groupby(['animal_id', 'model'])['avg_test_error'].agg(
    ['mean', 'std', 'count']
)

## 3. ANOVA Model Comparison

For each animal: 64 BE test errors vs 64 SC test errors.
One-way ANOVA tests whether the means differ. The model with
lower mean error wins (if p < 0.05).

In [ ]:
comparison_df = run_anova(long_df)
print(comparison_df[['animal_id', 'be_mean', 'sc_mean', 'p_value', 'winner']].to_string(index=False))

## 4. Per-Animal Comparison Plots

Violin plots of test error distributions + paired scatter.

In [ ]:
for aid in sorted(long_df['animal_id'].unique()):
    fig = plot_cv_comparison(
        long_df, comparison_df, aid,
        fit_target='UM', output_dir=str(OUTPUT_DIR),
    )
    plt.show()
    plt.close(fig)

## 5. Winner Summary

In [ ]:
fig = plot_winner_summary(comparison_df, output_dir=str(OUTPUT_DIR))
plt.show()

n_be = (comparison_df['winner'] == 'BE').sum()
n_sc = (comparison_df['winner'] == 'SC').sum()
n_ns = (comparison_df['winner'] == 'NS').sum()
print(f'BE: {n_be}, SC: {n_sc}, NS: {n_ns} / {len(comparison_df)} animals')

## 6. Best-Fit Parameters

For each animal, extract the winning model's parameters from the
seed with lowest test error.

In [ ]:
summary_df = build_summary_table(all_results, comparison_df)
summary_df

In [ ]:
for _, row in summary_df.iterrows():
    aid = row['animal_id']
    winner = row['winner']
    p = row['p_value']
    print(f'\n{aid}: winner = {winner} (p = {p:.2e})')
    print(f'  BE mean error = {row["be_mean_error"]:.5f}')
    print(f'  SC mean error = {row["sc_mean_error"]:.5f}')
    if winner in ('BE', 'SC'):
        param_cols = [c for c in summary_df.columns
                      if c.startswith('best_') and c != 'best_seed']
        print(f'  Best seed = {row.get("best_seed", "?")}')
        for col in param_cols:
            val = row.get(col, np.nan)
            if pd.notna(val):
                print(f'  {col.replace("best_", "")} = {val:.4f}')

## 7. Empirical vs Model Update Matrices

For each animal: load the raw behavioural data, compute the empirical
update matrix from expert sessions, simulate both models with their
best-fit parameters, and compare.

Uses `compute_update_matrix_from_sessions` (new architecture) for the
empirical UM, and `_simulate_um` from `analysis/grid_search` for the
model UMs.

In [ ]:
experiment = load_experiment(CONFIG_PATH)

# Select expert sessions per animal
expert_sessions = {}
for aid in sorted(all_results.keys()):
    try:
        animal = experiment.get_animal(aid)
        expert = select_sessions(
            animal, 'expert_uniform',
            min_accuracy=EXPERT_MIN_ACCURACY,
            last_fraction=EXPERT_LAST_FRACTION,
        )
        if len(expert) >= 3:
            expert_sessions[aid] = expert
            n_trials = sum(s.trials.valid_mask.sum() for s in expert)
            print(f'  {aid}: {len(expert)} sessions, ~{n_trials} trials')
        else:
            print(f'  {aid}: only {len(expert)} expert sessions — skipping')
    except Exception as e:
        print(f'  {aid}: SKIP — {e}')

In [ ]:
from analysis.grid_search import _sessions_to_arrays

for aid in sorted(expert_sessions.keys()):
    sessions = expert_sessions[aid]
    results_list = all_results[aid]

    # Empirical UM (new architecture)
    emp_um, _, emp_info = compute_update_matrix_from_sessions(
        sessions, method='pool', n_bins=N_BINS,
    )

    # Pool session arrays for model simulation
    data = _sessions_to_arrays(sessions)

    # Simulate best-fit model for each model type
    model_ums = {}
    model_errors = {}

    for model_name in ['BE', 'SC']:
        raw_params, seed = get_best_seed_params(results_list, model_name)
        if raw_params is None:
            continue

        named = format_params(model_name, raw_params)
        # Normalise param names for _simulate_um
        sp = named.get('sigma_noise', named.get('sigma_percep', 0.15))
        ar = named.get('A_repulsion', 0.10)

        if model_name == 'BE':
            p1 = named.get('eta_learning', 0.35)
            p2 = named.get('eta_relax', 0.12)
            p1_name, p2_name = 'eta_learning', 'eta_relax'
        else:
            p1 = named.get('gamma', 0.95)
            p2 = named.get('sigma_update', 0.30)
            p1_name, p2_name = 'gamma', 'sigma_update'

        try:
            model_um = _simulate_um(
                model_name,
                data['stimuli'], data['categories'],
                data['no_response'], data['not_blockstart'],
                sp, ar, p1, p2, p1_name, p2_name,
                seed, burn_in=1000, n_bins=N_BINS,
            )
            model_ums[model_name] = model_um
            model_errors[model_name] = matrix_error(model_um, emp_um)

            param_str = ', '.join(f'{k}={v:.3f}' for k, v in named.items())
            print(f'  {aid} {model_name}: MSE={model_errors[model_name]:.5f} | {param_str}')
        except Exception as e:
            print(f'  {aid} {model_name}: simulation failed — {e}')

    # Plot: empirical + both models
    if model_ums:
        winner_row = comparison_df[comparison_df['animal_id'] == aid]
        winner = winner_row['winner'].values[0] if len(winner_row) > 0 else '?'

        phases = {'Empirical': emp_um}
        annotations = {'Empirical': f'{emp_info["n_sessions"]}s pooled'}
        for mn, um in model_ums.items():
            err = model_errors[mn]
            marker = ' ★' if mn == winner else ''
            phases[f'{mn} (MSE={err:.4f}){marker}'] = um
            annotations[f'{mn} (MSE={err:.4f}){marker}'] = ''

        fig, axes = plot_phase_update_matrices(
            phases,
            suptitle=f'{aid}: winner = {winner}',
            annotations=annotations,
        )
        plt.show()

## 8. Parameter Distributions Across Seeds

For each animal, show how best-fit parameters vary across the 64
random seeds. Tight distributions = robust fit; broad = sensitive
to noise.

In [ ]:
param_df = extract_param_df(all_results)
print(f'Parameter DataFrame: {len(param_df)} rows')
param_df.head(10)

In [ ]:
for aid in sorted(all_results.keys()):
    fig = plot_param_distributions(
        param_df, aid, output_dir=str(OUTPUT_DIR),
    )
    plt.show()
    plt.close(fig)

## 9. Save

In [ ]:
long_df.to_csv(OUTPUT_DIR / 'cv_test_errors_long.csv', index=False)
comparison_df.to_csv(OUTPUT_DIR / 'cv_comparison_anova.csv', index=False)
summary_df.to_csv(OUTPUT_DIR / 'cv_summary.csv', index=False)
param_df.to_csv(OUTPUT_DIR / 'cv_best_params.csv', index=False)

print('Saved:')
for f in ['cv_test_errors_long.csv', 'cv_comparison_anova.csv',
          'cv_summary.csv', 'cv_best_params.csv']:
    print(f'  {OUTPUT_DIR / f}')

## Interpretation

**Clear winner (p < 0.05, consistent across seeds):** The winning model
better captures this animal's serial dependence structure during expert
performance. This classification carries forward to 4b (post-shift
parameter tracking) and 6a (opto predictions).

**Non-significant (NS):** Either both models fit equally well (both
capture the animal's behaviour), or neither fits well (flat update
matrix, weak serial dependence). Check the empirical UM — if it's
noisy/flat, the animal may not have strong enough serial dependence
to distinguish models.

**Parameter consistency across seeds:** If best-fit parameters vary
widely across seeds, the grid is too coarse or the cost surface has
multiple minima. Consider the coarse → fine grid refinement.

**Update matrix visual check:** The empirical UM should show either a
win-stay stripe (BE signature) or a more diffuse pattern (SC signature).
The winning model's simulated UM should visually match the empirical one.
If the ANOVA says BE wins but the UM doesn't look like a stripe, the
classification may be driven by a different feature of the UM.

**Next**: notebook 3b (SBI comparison), then 3c (consensus).